In [0]:
# ============================================
# 01. RUN SETUP NOTEBOOK
# ============================================

%run ./00_Setup.ipynb

In [0]:
# ============================================
# 02. DEFINE SILVER AND GOLD TABLES
# ============================================

SILVER_SCHEMA = f"{CATALOG}.silver"
GOLD_SCHEMA = f"{CATALOG}.gold"

# Silver source tables
SILVER_CUSTOMERS = f"{SILVER_SCHEMA}.customers"
SILVER_BOOKS = f"{SILVER_SCHEMA}.books"
SILVER_ORDERS = f"{SILVER_SCHEMA}.orders"
SILVER_ORDER_DETAILS = f"{SILVER_SCHEMA}.order_details"

# Gold reporting tables
GOLD_DAILY_SALES = f"{GOLD_SCHEMA}.daily_sales"
GOLD_CATEGORY_SALES = f"{GOLD_SCHEMA}.sales_by_category"
GOLD_CITY_SALES = f"{GOLD_SCHEMA}.sales_by_city"
GOLD_TOP_BOOKS = f"{GOLD_SCHEMA}.top_books"
GOLD_CUSTOMER_SUMMARY = f"{GOLD_SCHEMA}.customer_summary"
GOLD_ORDER_STATUS = f"{GOLD_SCHEMA}.order_status_summary"

print("Silver schema:", SILVER_SCHEMA)
print("Gold schema  :", GOLD_SCHEMA)

print("\nGold tables:")
print(GOLD_DAILY_SALES)
print(GOLD_CATEGORY_SALES)
print(GOLD_CITY_SALES)
print(GOLD_TOP_BOOKS)
print(GOLD_CUSTOMER_SUMMARY)
print(GOLD_ORDER_STATUS)

In [0]:
# ============================================
# 03. CREATE GOLD SCHEMA
# ============================================

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}
""")

print("Gold schema is ready.")

In [0]:
# ============================================
# 04. READ SILVER TABLES
# ============================================

from pyspark.sql import functions as F

silver_customers_df = spark.table(SILVER_CUSTOMERS)
silver_books_df = spark.table(SILVER_BOOKS)
silver_orders_df = spark.table(SILVER_ORDERS)
silver_order_details_df = spark.table(SILVER_ORDER_DETAILS)

print("Silver Customers:", silver_customers_df.count())
print("Silver Books    :", silver_books_df.count())
print("Silver Orders   :", silver_orders_df.count())
print("Silver Details  :", silver_order_details_df.count())

In [0]:
# ============================================
# 05. INSPECT SILVER ORDER DETAILS
# ============================================

silver_order_details_df.printSchema()

display(
    silver_order_details_df.limit(10)
)

In [0]:
# ============================================
# 06. OVERALL SALES KPIs
# ============================================

sales_kpis_df = (
    silver_order_details_df
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("quantity").alias("total_books_sold"),
        F.sum("total_amount").alias("total_revenue"),
        F.avg("total_amount").alias("average_order_value"),
        F.countDistinct("customer_id").alias("unique_customers")
    )
)

display(sales_kpis_df)

In [0]:
# ============================================
# 07. GOLD DAILY SALES
# ============================================

gold_daily_sales_df = (
    silver_order_details_df

    .withColumn(
        "order_date",
        F.to_date("order_timestamp")
    )

    .groupBy("order_date")

    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("quantity").alias("total_books_sold"),
        F.sum("total_amount").alias("total_revenue"),
        F.countDistinct("customer_id").alias("unique_customers")
    )

    .orderBy("order_date")
)

display(gold_daily_sales_df)

In [0]:
# ============================================
# 08. SAVE GOLD DAILY SALES
# ============================================

(
    gold_daily_sales_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_DAILY_SALES)
)

print("Created:", GOLD_DAILY_SALES)

In [0]:
# ============================================
# 09. GOLD SALES BY CATEGORY
# ============================================

gold_category_sales_df = (
    silver_order_details_df

    .groupBy("book_category")

    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("quantity").alias("total_books_sold"),
        F.sum("total_amount").alias("total_revenue"),
        F.avg("unit_price").alias("average_unit_price")
    )

    .orderBy(F.desc("total_revenue"))
)

display(gold_category_sales_df)

In [0]:
# ============================================
# 10. SAVE GOLD CATEGORY SALES
# ============================================

(
    gold_category_sales_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_CATEGORY_SALES)
)

print("Created:", GOLD_CATEGORY_SALES)

In [0]:
# ============================================
# 11. GOLD SALES BY CITY
# ============================================

gold_city_sales_df = (
    silver_order_details_df

    .groupBy("city")

    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.countDistinct("customer_id").alias("unique_customers"),
        F.sum("quantity").alias("total_books_sold"),
        F.sum("total_amount").alias("total_revenue")
    )

    .orderBy(F.desc("total_revenue"))
)

display(gold_city_sales_df)

In [0]:
# ============================================
# 12. SAVE GOLD CITY SALES
# ============================================

(
    gold_city_sales_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_CITY_SALES)
)

print("Created:", GOLD_CITY_SALES)

In [0]:
# ============================================
# 13. GOLD TOP-SELLING BOOKS
# ============================================

gold_top_books_df = (
    silver_order_details_df

    .groupBy(
        "book_id",
        "book_title",
        "book_category"
    )

    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("quantity").alias("total_quantity_sold"),
        F.sum("total_amount").alias("total_revenue")
    )

    .orderBy(
        F.desc("total_quantity_sold"),
        F.desc("total_revenue")
    )
)

display(gold_top_books_df.limit(20))

In [0]:
# ============================================
# 14. SAVE GOLD TOP BOOKS
# ============================================

(
    gold_top_books_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_TOP_BOOKS)
)

print("Created:", GOLD_TOP_BOOKS)

In [0]:
# ============================================
# 15. GOLD CUSTOMER SUMMARY
# ============================================

gold_customer_summary_df = (
    silver_order_details_df

    .groupBy(
        "customer_id",
        "first_name",
        "last_name",
        "email",
        "city"
    )

    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("quantity").alias("total_books_purchased"),
        F.sum("total_amount").alias("total_spent"),
        F.max("order_timestamp").alias("last_order_timestamp")
    )

    .orderBy(F.desc("total_spent"))
)

display(gold_customer_summary_df.limit(20))

In [0]:
# ============================================
# 16. SAVE GOLD CUSTOMER SUMMARY
# ============================================

(
    gold_customer_summary_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_CUSTOMER_SUMMARY)
)

print("Created:", GOLD_CUSTOMER_SUMMARY)

In [0]:
# ============================================
# 17. GOLD ORDER STATUS SUMMARY
# ============================================

gold_order_status_df = (
    silver_orders_df

    .groupBy("order_status")

    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("quantity").alias("total_books_sold"),
        F.sum("total_amount").alias("total_revenue")
    )

    .orderBy(F.desc("total_orders"))
)

display(gold_order_status_df)

In [0]:
# ============================================
# 18. SAVE GOLD ORDER STATUS SUMMARY
# ============================================

(
    gold_order_status_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_ORDER_STATUS)
)

print("Created:", GOLD_ORDER_STATUS)

In [0]:
# ============================================
# 19. VERIFY GOLD TABLES
# ============================================

GOLD_TABLES = [
    GOLD_DAILY_SALES,
    GOLD_CATEGORY_SALES,
    GOLD_CITY_SALES,
    GOLD_TOP_BOOKS,
    GOLD_CUSTOMER_SUMMARY,
    GOLD_ORDER_STATUS
]

for table_name in GOLD_TABLES:

    record_count = spark.table(table_name).count()

    print(f"Table: {table_name}")
    print(f"Records: {record_count}")
    print("-" * 60)

In [0]:
# ============================================
# 20. QUERY GOLD DAILY SALES
# ============================================

display(
    spark.sql(f"""
        SELECT *
        FROM {GOLD_DAILY_SALES}
        ORDER BY order_date DESC
    """)
)
#Category sales:
display(
    spark.sql(f"""
        SELECT *
        FROM {GOLD_CATEGORY_SALES}
        ORDER BY total_revenue DESC
    """)
)
#Top-selling books:
display(
    spark.sql(f"""
        SELECT *
        FROM {GOLD_TOP_BOOKS}
        ORDER BY total_quantity_sold DESC
        LIMIT 10
    """)
)
#Highest-spending customers:
display(
    spark.sql(f"""
        SELECT *
        FROM {GOLD_CUSTOMER_SUMMARY}
        ORDER BY total_spent DESC
        LIMIT 10
    """)
)

In [0]:
# ============================================
# 21. GOLD LAYER COMPLETION CHECK
# ============================================

print("GOLD LAYER SUMMARY")
print("=" * 60)

for table_name in GOLD_TABLES:
    print(
        f"{table_name}: "
        f"{spark.table(table_name).count()} records"
    )

print("=" * 60)
print("Gold aggregation notebook completed.")